# Video 08 - The agent loop

**You are here:** Video 8 of 13 - the Agent phase begins (see `../you-are-here.svg`). On the agent anatomy picture, today lights up **the LOOP**.

**The one idea:** one tool round is not enough. Put Videos 06-07 in a loop: ask the model, run tools, ask again - until it answers. **That loop is an agent.**

**By the end you can:**
1. Write a ~40-line agent that keeps going until the model is done.
2. Chain several tools to answer one question.
3. Explain why a step limit belongs in the loop from day one.

**Before you start:** attach this notebook to **Serverless**.


In [ ]:
import json
from datetime import date
from databricks.sdk import WorkspaceClient

client = WorkspaceClient().serving_endpoints.get_open_ai_client()
MODEL = "databricks-meta-llama-3-3-70b-instruct"
OTHER_MODEL = "databricks-claude-haiku-4-5"

def add_tool_results(messages, msg, functions):
    """Video 07 helper: append the request, run every call, string results, matching ids."""
    messages.append(msg.model_dump(exclude_none=True))                          # rule 1
    for call in msg.tool_calls:                                                  # rule 4
        args = json.loads(call.function.arguments or "{}")
        func = functions.get(call.function.name)
        result = func(**args) if func else f"error: unknown tool {call.function.name}"
        content = result if isinstance(result, str) else json.dumps(result)      # rule 3
        messages.append({"role": "tool", "tool_call_id": call.id, "content": content})  # rule 2

print("Connected.")


## Why one round isn't enough

In Video 06, asking about **two cities** often needed more than one tool round on Llama (one city per reply). Below: do exactly one round, then call again - and see whether you have a final answer yet.


In [ ]:
def get_weather(city: str) -> str:
    return {"perth": "sunny, 31C", "london": "rainy, 12C", "tokyo": "cloudy, 18C"}.get(city.lower(), "unknown city")

WEATHER_TOOL = {"type": "function", "function": {
    "name": "get_weather", "description": "Get the current weather for a city.",
    "parameters": {"type": "object", "properties": {"city": {"type": "string"}}, "required": ["city"]}}}

messages = [{"role": "user", "content": "What's the weather in Perth and in Tokyo?"}]
msg = client.chat.completions.create(model=MODEL, messages=messages, tools=[WEATHER_TOOL]).choices[0].message
print("After first ask:", "tool_calls" if msg.tool_calls else "final answer")
print("  requested:", [c.function.name + "(" + c.function.arguments + ")" for c in (msg.tool_calls or [])])

if msg.tool_calls:
    add_tool_results(messages, msg, {"get_weather": get_weather})
    msg2 = client.chat.completions.create(model=MODEL, messages=messages, tools=[WEATHER_TOOL]).choices[0].message
    print("After one round:", "tool_calls" if msg2.tool_calls else "final answer")
    if msg2.tool_calls:
        print("  still wants:", [c.function.name + "(" + c.function.arguments + ")" for c in msg2.tool_calls])
        print("-> One round was not enough. A loop would keep going.")
    else:
        print("  answer:", msg2.content)


## Define the tools (plain Python)

Three ordinary functions - an order lookup, a calculator, and today's date. The agent will chain them.


In [ ]:
ORDERS = {42: {"customer": "Alice", "total": 250.0}, 7: {"customer": "Bob", "total": 80.0}}

def get_order_total(customer_id: int):
    order = ORDERS.get(int(customer_id))
    return order["total"] if order else "customer not found"

def calculator(expression: str):
    # Safety: only allow digits and arithmetic symbols, never arbitrary code
    if not set(expression) <= set("0123456789+-*/(). "):
        return "error: only basic arithmetic is allowed"
    return eval(expression, {"__builtins__": {}})

def get_today():
    return date.today().isoformat()

TOOLS = {"get_order_total": get_order_total, "calculator": calculator, "get_today": get_today}
print("Tools ready:", list(TOOLS))


In [ ]:
def tool_spec(name, description, properties=None, required=None):
    """Small helper so tool descriptions are less verbose."""
    return {"type": "function", "function": {
        "name": name, "description": description,
        "parameters": {"type": "object", "properties": properties or {}, "required": required or []}}}

TOOL_SPECS = [
    tool_spec("get_order_total", "Look up the order total in dollars for a customer id.",
              {"customer_id": {"type": "integer"}}, ["customer_id"]),
    tool_spec("calculator", "Evaluate a basic arithmetic expression, e.g. '250 * 0.15'.",
              {"expression": {"type": "string"}}, ["expression"]),
    tool_spec("get_today", "Get today's date in ISO format."),
]
print(json.dumps(TOOL_SPECS, indent=2))


## The agent loop

Ask the model. If it wants tools, run them (Video 07 rules) and ask again. If not, return the answer. Stop if you hit `max_steps`.


In [ ]:
def run_agent(question, max_steps=5, verbose=True, model=MODEL):
    """LLM + tools + loop + memory. About 40 lines. That's an agent."""
    messages = [
        {"role": "system", "content": "You are a helpful assistant. Use the tools when you need facts or maths."},
        {"role": "user", "content": question},
    ]
    for step in range(1, max_steps + 1):                       # safety: never loop forever
        response = client.chat.completions.create(model=model, messages=messages, tools=TOOL_SPECS)
        msg = response.choices[0].message

        if not msg.tool_calls:                                 # no tools requested -> final answer
            if verbose:
                print(f"  step {step}: final answer")
            return msg.content

        if verbose:
            for tc in msg.tool_calls:
                print(f"  step {step}: {tc.function.name}({tc.function.arguments})")
        add_tool_results(messages, msg, TOOLS)                 # all four golden rules

    return "Stopped: reached max_steps without a final answer."

print("run_agent defined.")


## Your first real agent

**Predict first:** for *"What's 15% of the order total for customer 42?"* - which tools, in what order?


In [ ]:
print(run_agent("What's 15% of the order total for customer 42?"))


In [ ]:
questions = [
    "What is today's date?",
    "How much more did customer 42 spend than customer 7?",
    "What is the order total for customer 999?",   # tool returns 'not found' - how does the agent cope?
    "Tell me a fun fact about Python.",            # no tool needed
]
for q in questions:
    print("Q:", q)
    print("A:", run_agent(q), "\n")


## Design choices: why our loop looks the way it does

Many tutorials write `while True:` and check `finish_reason == "tool_calls"`. Ours makes a few deliberate changes:

| Tutorial style | Our style | Why |
| --- | --- | --- |
| `while True:` | `for step in range(max_steps):` | A confused model can request tools forever. A step limit is your first **guardrail**. |
| `if finish_reason == "tool_calls"` | `if not msg.tool_calls` | Checks what the model *actually returned*. A reply cut off with `finish_reason == "length"` is not a finished answer. |
| Hard-code one tool | `TOOLS[name](**args)` | **Multi-tool routing**: run whichever function the model picked. |

Everything else - append the assistant message, run each tool, append `tool` messages with matching ids, call again - is Video 07 inside a loop. Same dance, until done.


## Cost meter: each step is a model call

Agents multiply cost by chain length. Count the calls for the 15% question.


In [ ]:
def run_agent_counting(question, max_steps=5, model=MODEL):
    """Same loop, but returns (answer, n_calls, total_tokens)."""
    messages = [
        {"role": "system", "content": "You are a helpful assistant. Use the tools when you need facts or maths."},
        {"role": "user", "content": question},
    ]
    n_calls = 0
    total_tokens = 0
    for step in range(1, max_steps + 1):
        response = client.chat.completions.create(model=model, messages=messages, tools=TOOL_SPECS)
        n_calls += 1
        total_tokens += response.usage.total_tokens
        msg = response.choices[0].message
        if not msg.tool_calls:
            return msg.content, n_calls, total_tokens
        add_tool_results(messages, msg, TOOLS)
    return "Stopped: max_steps", n_calls, total_tokens

PRICE_PER_1M = 0.50   # EXAMPLE price in USD - replace with your real rate
answer, n_calls, total_tokens = run_agent_counting("What's 15% of the order total for customer 42?")
print("Answer:", answer)
print(f"Model calls: {n_calls}")
print(f"Total tokens (all calls): {total_tokens}")
print(f"Example cost: ${total_tokens / 1_000_000 * PRICE_PER_1M:.4f}")
print("-> Each loop iteration = one model call. Longer chains cost more.")


## What could go wrong? No step limit

Set `max_steps=1` on a question that needs two tools. The agent stops mid-job. Without any limit, the opposite risk: a model that keeps calling tools burns tokens until you kill the notebook.


In [ ]:
print(run_agent("What's 15% of the order total for customer 42?", max_steps=1))
print("-> Stopped after one tool step. The calculator never ran. max_steps is your first seatbelt.")


## Recap

* An **agent** = LLM + tools + loop + memory.
* One tool round is not enough - keep going until `tool_calls` is empty.
* Use a **step limit** from day one.
* Check `msg.tool_calls`, route with a `TOOLS` dict, reuse Video 07's golden rules.

**Exercises:** see `../exercises.md` (about 15 minutes). **Quiz:** `../quiz.md`. **Cheatsheet:** `../cheatsheet.md`.

**Next:** Video 09 - Keeping agents safe (more guardrails, errors as results, retries).
